In [2]:
# Welcome to your new notebook
# Type here in the cell editor to add code!


StatementMeta(, 3ba087b0-304e-428d-b3a5-ca42a42fd272, 4, Finished, Available, Finished, False)

##### <mark>**STREAMING INGESTION**</mark>

- **STREAMING INGESTION SIMULATION**

<mark>**Goal:**</mark>
- **Simulate arriving order-event files**
- **Read them as a stream**
- **Land them into Bronze Delta table**

###### <mark>**Step 2 Create batch_1 sample data**</mark>

In [1]:
from pyspark.sql import Row

sample_events_batch_1 = [
    Row(order_id=900001, customer_id=101, product_id=2001, event_type="order_placed",   event_time="2026-03-31 10:00:00", amount=120.50, status="created"),
    Row(order_id=900002, customer_id=102, product_id=2002, event_type="order_placed",   event_time="2026-03-31 10:01:00", amount=80.00,  status="created"),
    Row(order_id=900003, customer_id=103, product_id=2003, event_type="payment_success", event_time="2026-03-31 10:02:00", amount=220.00, status="paid"),
    Row(order_id=900004, customer_id=104, product_id=2004, event_type="order_placed",   event_time="2026-03-31 10:03:00", amount=55.75,  status="created")
]

batch_1_df = spark.createDataFrame(sample_events_batch_1)
display(batch_1_df)

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 3, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ff26d1f5-69c5-49f4-ab96-34b1be4d7ace)

###### <mark>**Step 3: Write batch_1 into the input folder**</mark>

In [2]:
batch_1_df.write \
    .mode("append") \
    .json("Files/streaming/order_events_input")

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 4, Finished, Available, Finished, False)

###### <mark>**Step 4: Define streaming schema**</mark>

In [3]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, DoubleType

order_event_schema = StructType([
    StructField("order_id", LongType(), True),
    StructField("customer_id", LongType(), True),
    StructField("product_id", LongType(), True),
    StructField("event_type", StringType(), True),
    StructField("event_time", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("status", StringType(), True)
])

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 5, Finished, Available, Finished, False)

<mark>**validate Steaming Schema**</mark>

###### <mark>**Step 5: Read input folder as streaming source**</mark>

In [4]:
streaming_orders_df = (
    spark.readStream
    .schema(order_event_schema)
    .json("Files/streaming/order_events_input")
)

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 6, Finished, Available, Finished, False)

###### <mark>**Step 6: Validate that stream started**</mark>

In [5]:
streaming_orders_df.printSchema()

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 7, Finished, Available, Finished, False)

root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product_id: long (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)



###### <mark>**Step 7: Start Bronze streaming write**</mark>
**Now create a second batch to simulate streaming arrival.**

In [6]:
bronze_stream_query = (
    streaming_orders_df.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", "Files/streaming/checkpoints/bronze_order_events_stream")
    .trigger(processingTime="10 seconds")
    .toTable("bronze_order_events_stream")
)

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 8, Finished, Available, Finished, False)

###### <mark>**Step 8: Check stream status**</mark>

In [7]:
print("Is streaming active?", bronze_stream_query.isActive)
print(bronze_stream_query.status)
print(bronze_stream_query.lastProgress)

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 9, Finished, Available, Finished, False)

Is streaming active? True
{'message': 'Waiting for next trigger', 'isDataAvailable': False, 'isTriggerActive': False}
{'id': '2bcfde3a-b3dc-467e-91ee-769e608eb382', 'runId': '80c3a96f-f08a-4b4e-9328-b01a8af97090', 'name': None, 'timestamp': '2026-04-01T06:09:30.003Z', 'batchId': 1, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0, 'durationMs': {'latestOffset': 134, 'triggerExecution': 134}, 'stateOperators': [], 'sources': [{'description': 'FileStreamSource[abfss://eaf0c26f-0ff8-484b-907a-fbe7a597851a@onelake.dfs.fabric.microsoft.com/8702ce55-bae0-44d5-b9b0-0018d638c4c0/Files/streaming/order_events_input]', 'startOffset': {'logOffset': 0}, 'endOffset': {'logOffset': 0}, 'latestOffset': None, 'numInputRows': 0, 'inputRowsPerSecond': 0.0, 'processedRowsPerSecond': 0.0}], 'sink': {'description': 'DeltaSink[abfss://eaf0c26f-0ff8-484b-907a-fbe7a597851a@onelake.dfs.fabric.microsoft.com/8702ce55-bae0-44d5-b9b0-0018d638c4c0/Tables/dbo/bronze_order_events_stream]', '

###### <mark>**validate Bronze table**</mark>

In [8]:
spark.sql("""
SELECT *
FROM bronze_order_events_stream
ORDER BY order_id
""").show(truncate=False)

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 10, Finished, Available, Finished, False)

+--------+-----------+----------+---------------+-------------------+------+-------+
|order_id|customer_id|product_id|event_type     |event_time         |amount|status |
+--------+-----------+----------+---------------+-------------------+------+-------+
|900001  |101        |2001      |order_placed   |2026-03-31 10:00:00|120.5 |created|
|900001  |101        |2001      |order_placed   |2026-03-31 10:00:00|120.5 |created|
|900002  |102        |2002      |order_placed   |2026-03-31 10:01:00|80.0  |created|
|900002  |102        |2002      |order_placed   |2026-03-31 10:01:00|80.0  |created|
|900003  |103        |2003      |payment_success|2026-03-31 10:02:00|220.0 |paid   |
|900003  |103        |2003      |payment_success|2026-03-31 10:02:00|220.0 |paid   |
|900004  |104        |2004      |order_placed   |2026-03-31 10:03:00|55.75 |created|
|900004  |104        |2004      |order_placed   |2026-03-31 10:03:00|55.75 |created|
+--------+-----------+----------+---------------+----------------

###### <mark>**Step 9: Create batch_2 sample data**</mark>

In [9]:
sample_events_batch_2 = [
    Row(order_id=900005, customer_id=105, product_id=2005, event_type="order_placed",   event_time="2026-03-31 10:05:00", amount=310.25, status="created"),
    Row(order_id=900006, customer_id=106, product_id=2006, event_type="payment_success", event_time="2026-03-31 10:06:00", amount=99.99,  status="paid"),
    Row(order_id=900007, customer_id=107, product_id=2007, event_type="order_placed",   event_time="2026-03-31 10:07:00", amount=145.10, status="created")
]

batch_2_df = spark.createDataFrame(sample_events_batch_2)
display(batch_2_df)

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5a0d59c0-1e92-426a-ace9-338d448802ab)

###### <mark>**Step 10: Append batch_2 into the same input folder**</mark>

In [10]:
batch_2_df.write \
    .mode("append") \
    .json("Files/streaming/order_events_input")

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 12, Finished, Available, Finished, False)

###### <mark>**Step 11: Validate batch_2 landed**</mark>

In [11]:
spark.sql("""
SELECT *
FROM bronze_order_events_stream
ORDER BY order_id
""").show(truncate=False)

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 13, Finished, Available, Finished, False)

+--------+-----------+----------+---------------+-------------------+------+-------+
|order_id|customer_id|product_id|event_type     |event_time         |amount|status |
+--------+-----------+----------+---------------+-------------------+------+-------+
|900001  |101        |2001      |order_placed   |2026-03-31 10:00:00|120.5 |created|
|900001  |101        |2001      |order_placed   |2026-03-31 10:00:00|120.5 |created|
|900002  |102        |2002      |order_placed   |2026-03-31 10:01:00|80.0  |created|
|900002  |102        |2002      |order_placed   |2026-03-31 10:01:00|80.0  |created|
|900003  |103        |2003      |payment_success|2026-03-31 10:02:00|220.0 |paid   |
|900003  |103        |2003      |payment_success|2026-03-31 10:02:00|220.0 |paid   |
|900004  |104        |2004      |order_placed   |2026-03-31 10:03:00|55.75 |created|
|900004  |104        |2004      |order_placed   |2026-03-31 10:03:00|55.75 |created|
|900005  |105        |2005      |order_placed   |2026-03-31 10:05

###### <mark>**Step 12: Optional: stop the stream after testing**</mark>

In [12]:
bronze_stream_query.stop()

StatementMeta(, 4318cc53-03ca-4ce7-9b95-b4899aba67b8, 14, Finished, Available, Finished, False)

In [1]:
spark.sql("""
SELECT COUNT(*) FROM bronze_order_events_stream
""").show()

StatementMeta(, ae46ce18-0fda-43cb-9f00-15ac4575e011, 3, Finished, Available, Finished, False)

+--------+
|count(1)|
+--------+
|      11|
+--------+



In [2]:
spark.sql("""
SELECT * FROM bronze_order_events_stream
""").show()

StatementMeta(, ae46ce18-0fda-43cb-9f00-15ac4575e011, 4, Finished, Available, Finished, False)

+--------+-----------+----------+---------------+-------------------+------+-------+
|order_id|customer_id|product_id|     event_type|         event_time|amount| status|
+--------+-----------+----------+---------------+-------------------+------+-------+
|  900005|        105|      2005|   order_placed|2026-03-31 10:05:00|310.25|created|
|  900004|        104|      2004|   order_placed|2026-03-31 10:03:00| 55.75|created|
|  900004|        104|      2004|   order_placed|2026-03-31 10:03:00| 55.75|created|
|  900001|        101|      2001|   order_placed|2026-03-31 10:00:00| 120.5|created|
|  900003|        103|      2003|payment_success|2026-03-31 10:02:00| 220.0|   paid|
|  900006|        106|      2006|payment_success|2026-03-31 10:06:00| 99.99|   paid|
|  900003|        103|      2003|payment_success|2026-03-31 10:02:00| 220.0|   paid|
|  900001|        101|      2001|   order_placed|2026-03-31 10:00:00| 120.5|created|
|  900007|        107|      2007|   order_placed|2026-03-31 10:07